# 06 - Split (portfolios)

Takes the odds capture form written by `05_Run`, once the prices are in, and
turns it into a set of portfolios to choose between.

The pipeline, in order:

1. **Best price** -- `O = max` over the books in `staking.BOOKS`. A proposition
   priced by no book is dropped. `Book` names the winner, or every winner on a
   tie.
2. **Edge** -- `E = P x O`. Above 1 the book pays more than the model says it
   should.
3. **Discard `E < 1`.** An event can lose everything here; that is normal.
4. **Remove dominated propositions, within each event** -- anything another beats
   on *both* `P` and `O`. Not `P` and `E`: `E` is `P x O`, so comparing against it
   counts the probability twice. Exact ties on both survive.
5. **Search the portfolios** -- every selection of one proposition from each of
   any subset of events, scored under both the `1/E` split and the
   minimum-variance split.
6. **Keep the undominated ones** -- nothing else beats them on expected return,
   variance *and* probability of profit at once.

There is no step 7 and no final list. At portfolio level the three criteria trade
against each other and there is no single best answer, so what comes out is a set
to filter, in the notebook or on the workbook's Query sheet.

This notebook contains no logic of its own; everything lives in `fpp.portfolio`.

In [1]:
import fpp
import numpy as np
import pandas as pd

from fpp import portfolio as pf

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
print("fpp", fpp.__version__)

fpp 0.1.0


## 1. Inputs

`ODDS_FILE` **resolves to the newest form on disk** rather than being typed. It
used to be a hardcoded date, and that is exactly how a run two days later
consumed a two-day-old form: fixtures that were no longer on, priced by a model
that had since been retuned, and nothing anywhere to notice. The form carries its
own probabilities and `06_Split` reads nothing else, so there is no second source
that could have disagreed.

`check_form_is_current` raises if the file is not the newest one. Pin a specific
form deliberately if you mean to re-price an old one.

`STAKE` seeds cell B1 of the calculator -- it stays editable in Excel afterwards,
and every percentage is independent of it, so this number only sets the cash
column.

`LEG_VAR` is **how many events you are willing to leave out**, and it is the
skipping switch. `0` requires one bet from every qualifying event, which is the
only setting whose answer is provable -- the space collapses to something that
enumerates, so every portfolio is scored rather than searched. Each rung above
that opens the search one event wider.

It is deliberately *relative*, and that is the whole point of the control. How
many events qualify is not knowable when you set it: a 45-fixture weekend might
yield 43 events or 37, depending on which markets got priced and which cleared
`e >= 1`. Typing an absolute floor against a guess of 43 quietly becomes "skip
nothing" if only 38 qualify, and asks for the impossible if 36 do. `LEG_VAR`
says the thing actually meant and resolves against whatever the form turns out
to hold.

Keeping it small is worth doing, and not only for the reason below. Every
variance and probability figure here is computed *assuming the model
probabilities are right*; nothing accounts for them being wrong. A two-leg
portfolio can read 95% profitability while resting its whole edge on one
proposition being accurate to a point or two. Spreading over more events is the
only defence against that, and it is the one risk the numbers omit.

It is also the cheapest runtime lever there is. The search itself takes the same
time whatever the floor, but what comes out of it does not -- on a 19-event form,
431,244 states at no floor against 184,250 at `LEG_VAR = 5` and 30,593 at `0` --
and everything downstream is priced per portfolio.

In [2]:
ODDS_FILE = fpp.report.latest_form()      # newest odds_input_*.xlsx; pin a path to override
STAKE     = 220.0

LEG_VAR   = 5      # how many qualifying events a portfolio may leave out.
                   # 0 = one bet from every event, no skipping (and provable).
                   # Resolves against however many events actually qualify.

WRITE_WORKBOOK = False   # the Excel portfolio workbook. Superseded by the Edge
                         # Book app below; kept because it costs one line to keep
                         # and 5.7 MB a day to write.

## 2. Read the form

In [3]:
# Refuses a form that is not the newest on disk -- see the note above.
meta = fpp.report.check_form_is_current(ODDS_FILE)
print(f"form      : {meta['path'].name}")
print(f"generated : {meta['generated']}  from {meta['source']}")
print(f"fixtures  : {meta['fixtures']}")

filled = fpp.report.read_filled(ODDS_FILE)
display(filled.head(8))

form      : odds_input_2026-09-11.xlsx
generated : 2026-09-11  from predictions_2026-09-11.xlsx
fixtures  : 24
Read 1776 propositions from odds_input_2026-09-11.xlsx (generated 2026-09-11 from predictions_2026-09-11.xlsx); 835 priced


,sheet_code,label,p,b365,paddypower,tenbet,boylesports,betmgm,virginbet,date,league,home_team,away_team
0,D1-01,Goals - Union Berlin - Over 0.5,0.815981,1.200,1.142857,1.181,1.17,1.19,1.10,2026-09-11,Bundesliga,Union Berlin,Schalke 04
1,D1-01,Goals - Union Berlin - Over 1.5,0.504489,2.000,2.000000,1.930,1.87,1.92,1.66,2026-09-11,Bundesliga,Union Berlin,Schalke 04
2,D1-01,Goals - Union Berlin - Over 2.5,0.240855,4.000,4.000000,3.950,3.80,3.90,3.10,2026-09-11,Bundesliga,Union Berlin,Schalke 04
3,D1-01,Goals - Union Berlin - Over 3.5,0.092103,11.000,9.500000,NaN,9.50,8.00,6.40,2026-09-11,Bundesliga,Union Berlin,Schalke 04
4,D1-01,Goals - Schalke 04 - Over 0.5,0.633871,1.222,1.222222,1.250,1.20,1.22,1.26,2026-09-11,Bundesliga,Union Berlin,Schalke 04
5,D1-01,Goals - Schalke 04 - Over 1.5,0.265996,2.100,2.200000,2.200,2.10,2.08,2.28,2026-09-11,Bundesliga,Union Berlin,Schalke 04
6,D1-01,Goals - Schalke 04 - Over 2.5,0.081181,4.500,5.000000,4.750,4.50,4.33,5.10,2026-09-11,Bundesliga,Union Berlin,Schalke 04
7,D1-01,Goals - Schalke 04 - Over 3.5,0.019282,13.000,10.500000,NaN,12.00,9.00,13.00,2026-09-11,Bundesliga,Union Berlin,Schalke 04


## 3. Qualify

Watch the funnel. `(P, O)` dominance prunes far less than the old `(P, E)` rule
did -- `P` and `O` are close to inverses of each other, so they correlate
strongly negatively and mutual dominance is rare. That is the rule working: the
survivors are genuinely incomparable, and choosing between them is what the
search is for.

In [4]:
qualified = pf.qualify(filled)
options   = pf.event_options(qualified)
min_legs  = options.min_legs_for(LEG_VAR)   # the same resolution `pf.search` will do

print(f"priced           : {int(filled[list(fpp.staking.BOOK_COLUMNS)].notna().any(axis=1).sum())}")
print(f"E >= 1, undominated: {len(qualified)} across {options.n_events} events")
print(f"options per event  : {options.sizes.tolist()}")
print(f"leg range          : {min_legs}-{options.n_events} legs  (LEG_VAR = {LEG_VAR})")
print(f"search space       : {options.space(min_legs, None):,.4g} portfolios")

priced           : 835
E >= 1, undominated: 104 across 21 events
options per event  : [4, 7, 2, 4, 3, 2, 4, 9, 4, 6, 3, 10, 4, 3, 4, 2, 4, 10, 8, 6, 5]
leg range          : 16-21 legs  (LEG_VAR = 5)
search space       : 2.72e+15 portfolios


## 4. Search

Below `EXHAUSTIVE_MAX` every portfolio is enumerated; above it the dynamic
program walks the space instead, keeping the `(expected return, variance)`
frontier exactly and a band around it. `info["mode"]` says which happened, because
"all of them" and "the ones the search reached" are different claims.

In [5]:
res = pf.search(filled, leg_var=LEG_VAR)

info, scored = res["info"], res["scored"]
print(f"mode            : {info['mode']}  ({info['found']:,} reached -> pool {info['pool']:,})")
print(f"legs            : {info['min_legs']}-{info['max_legs']}  (leg_var = {info['leg_var']})")
print(f"rows scored     : {len(scored):,}")
print(f"undominated     : {int(scored['undominated'].sum()):,}")

mode            : searched  (183,927 reached -> pool 100,000)
legs            : 16-21  (leg_var = 5)
rows scored     : 100,000
undominated     : 2,360


## 5. The undominated set

Every row here is beaten by nothing else on all three of expected return,
variance and probability of profit. The extremes are worth looking at first --
they are what the trade-off actually costs.

In [6]:
# Read off `scored` rather than listed: `staking.THRESHOLDS` is meant to be tuned,
# and naming the columns by hand meant trimming it broke this cell rather than
# just narrowing the table. `median_leg_p` sits next to `legs` because the two
# answer the same question -- how many bets, and how likely each one is.
THRESH = [c for c in scored.columns if c.startswith("p_over_")]
VIEW = ["id", "split", "legs", "median_leg_p", "pct_expected_return", "pct_sd", *THRESH]
FMT  = {"pct_expected_return": "{:.2%}", "pct_sd": "{:.2%}",
        "median_leg_p": "{:.1%}", **{c: "{:.2%}" for c in THRESH}}

undominated = scored[scored["undominated"]]
if undominated.empty:
    print("Nothing survived -- either nothing was priced, or no price beat the model.")
else:
    for title, sub in (
        ("highest probability of profit", undominated.nlargest(5, "p_over_100")),
        ("highest expected return",       undominated.nlargest(5, "pct_expected_return")),
        ("lowest spread",                 undominated.nsmallest(5, "pct_sd")),
        ("likeliest legs",                undominated.nlargest(5, "median_leg_p")),
    ):
        print(f"\n--- {title} ---")
        display(sub[VIEW].style.format(FMT).hide(axis="index"))


--- highest probability of profit ---


id,split,legs,median_leg_p,pct_expected_return,pct_sd,p_over_90,p_over_100,p_over_110
45229,Growth,18,43.0%,142.00%,46.06%,86.24%,81.11%,74.83%
60975,Growth,19,39.6%,141.65%,45.72%,86.20%,81.10%,74.68%
60852,Growth,19,38.0%,142.13%,46.26%,86.27%,81.10%,74.85%
45247,Growth,18,41.0%,141.76%,45.86%,86.17%,81.09%,74.72%
92328,Growth,21,35.0%,142.15%,46.28%,86.24%,81.09%,74.85%



--- highest expected return ---


id,split,legs,median_leg_p,pct_expected_return,pct_sd,p_over_90,p_over_100,p_over_110
281,Growth,16,7.4%,165.62%,210.37%,46.94%,45.95%,43.81%
277,Growth,16,7.4%,163.84%,207.74%,43.32%,41.29%,38.88%
191,Growth,16,7.9%,160.76%,207.32%,42.26%,42.06%,41.32%
200,Growth,16,7.9%,160.71%,207.14%,42.25%,42.02%,41.30%
192,Growth,16,7.9%,160.62%,207.07%,42.05%,41.97%,41.52%



--- lowest spread ---


id,split,legs,median_leg_p,pct_expected_return,pct_sd,p_over_90,p_over_100,p_over_110
99971,Growth,21,51.1%,117.81%,21.61%,89.25%,79.68%,65.85%
99993,Growth,21,51.1%,117.99%,21.71%,89.28%,79.77%,66.02%
99975,Growth,21,51.1%,118.04%,21.78%,89.25%,79.76%,66.03%
99967,Growth,21,51.1%,118.05%,21.78%,89.27%,79.77%,66.06%
99998,Growth,21,51.1%,118.07%,21.80%,89.28%,79.78%,66.08%



--- likeliest legs ---


id,split,legs,median_leg_p,pct_expected_return,pct_sd,p_over_90,p_over_100,p_over_110
99946,Growth,21,51.2%,118.72%,22.30%,89.32%,80.06%,66.69%
84123,Growth,20,51.1%,119.15%,22.71%,89.30%,80.17%,67.02%
84126,Growth,20,51.1%,119.12%,22.65%,89.33%,80.16%,67.02%
84121,Growth,20,51.1%,118.15%,21.95%,89.20%,79.74%,66.08%
100000,Growth,21,51.1%,119.02%,22.48%,89.41%,80.24%,67.01%


## 6. Filter

`filter_portfolios` takes the same criteria as the workbook's Query sheet, as
`min_<column>` / `max_<column>`. Leave one out and that constraint is dropped.
There is no right answer here -- change the numbers until the shape of the
outcome is one you want.

In [7]:
picked = pf.filter_portfolios(
    scored,
    min_pct_expected_return=1.05,
    min_p_over_100=0.70,
    max_pct_sd=0.25,
    # The ledger's first settled slate ranked `median_leg_p` +0.72 against realised
    # return and `pct_expected_return` -0.76. One Saturday, so this is left off
    # rather than switched on -- but it is the constraint that evidence points at.
    # min_median_leg_p=0.55,
    sort_by="p_over_100",
)
print(f"{len(picked)} portfolios match")
display(picked[VIEW].head(15).style.format(FMT).hide(axis="index"))

141 portfolios match


id,split,legs,median_leg_p,pct_expected_return,pct_sd,p_over_90,p_over_100,p_over_110
99697,Growth,21,47.0%,120.20%,23.67%,89.27%,80.39%,67.76%
99660,Growth,21,47.0%,120.12%,23.59%,89.28%,80.38%,67.72%
99657,Growth,21,45.9%,120.23%,23.73%,89.25%,80.38%,67.78%
99596,Growth,21,48.2%,120.11%,23.59%,89.26%,80.36%,67.70%
99680,Growth,21,47.0%,120.33%,23.86%,89.21%,80.36%,67.78%
99207,Growth,21,45.9%,121.25%,24.89%,89.10%,80.36%,68.12%
99617,Growth,21,45.9%,120.15%,23.65%,89.25%,80.36%,67.72%
99166,Growth,21,45.9%,121.17%,24.81%,89.10%,80.34%,68.06%
99565,Growth,21,47.0%,120.36%,23.96%,89.18%,80.33%,67.79%
99976,Growth,21,50.6%,119.49%,22.92%,89.37%,80.33%,67.34%


## 7. One portfolio, as bets to place

Point this at any `id` from the tables above. `Book` travels with each leg
because an edge you cannot find again is not actionable.

In [8]:
if not scored.empty:
    row = (picked if len(picked) else undominated if len(undominated) else scored).iloc[0]
    bets = pf.legs(res["picks"], res["options"], int(row["combo"]), row["split"], total=STAKE)
    print(f"portfolio {int(row['id'])}  ({row['split']} split, {int(row['legs'])} legs)")
    print(f"  expected return {row['pct_expected_return']:.2%}   "
          f"sd {row['pct_sd']:.2%}   P(profit) {row['p_over_100']:.2%}")
    display(bets.style.format({"p": "{:.2%}", "o": "{:.2f}", "e": "{:.4f}",
                               "stake": "{:,.2f}"}).hide(axis="index"))

portfolio 99697  (Growth split, 21 legs)
  expected return 120.20%   sd 23.67%   P(profit) 80.39%


event,fixture,label,p,o,book,e,stake
D1-01,Union Berlin vs Schalke 04,Corners - Union Berlin - Over 6.5,35.01%,3.00,BoyleSports,1.0504,3.62
D1-02,Augsburg vs Bayer Leverkusen,Corners - Augsburg - Over 3.5,71.96%,1.61,Bet365,1.1622,20.53
D1-04,FC Cologne vs Werder Bremen,Corners - FC Cologne - Over 6.5,42.33%,2.60,BoyleSports,1.1006,5.79
D1-06,Hoffenheim vs VfB Stuttgart,Corners - Hoffenheim - Over 6.5,44.54%,2.60,BoyleSports,1.1582,7.80
D1-07,Mainz 05 vs Eintracht Frankfurt,Corners - Mainz 05 - Over 5.5,51.10%,2.00,Bet365,1.0219,3.46
E0-01,Aston Villa vs Nottingham Forest,Goals - Aston Villa - Over 1.5,47.05%,2.20,Bet365,1.0351,3.89
E0-02,Bournemouth vs Brentford,Goals - Bournemouth - Over 1.5,50.63%,2.05,10bet,1.0379,4.30
E0-03,Chelsea vs Hull,Corners - Hull - Over 2.5,59.12%,2.20,Paddy Power,1.3006,17.47
E0-04,Crystal Palace vs Ipswich,Goals - Ipswich - Over 1.5,38.02%,3.25,Bet365,1.2357,7.80
E0-05,Liverpool vs Fulham,Corners - Fulham - Over 4.5,43.61%,2.63,BoyleSports,1.1470,7.31


## 8. Write the Edge Book

Two JSON files and one page. `portfolios.json` carries the **undominated set
only** -- 4,940 of 200,000 scored rows on a real form -- with each portfolio's
legs as an array of proposition ids rather than duplicated rows, which is what
lets a 5.7 MB workbook become roughly 3 MB of data.

`write_edge_book` fuses that with the `predictions.json` written by `05_Run` and
the app source in `app/edge-book/` into one self-contained HTML file. Open it
directly -- there is no server and nothing to install. It refuses to build if the
two JSON files are from different runs, because a page pairing yesterday's
fixtures with today's portfolios would open looking perfectly fine.

Set `WRITE_WORKBOOK = True` in section 1 to also write the old Excel workbook.

Where it all lands: the page goes to the `Outputs` root and pushes yesterday's
into `Outputs/Archive/edge_book/`, and the JSON goes to `Outputs/Data/`. The root
holds one predictions workbook, one odds form and one page — which is what lets
`latest_form()` and `latest_json()` answer "the current one" by position rather
than by reading dates off a listing.

`ledger.capture` writes this run into `Outputs/Analysis/Pending/` on the way
past. That has to happen here rather than in `07_Analysis`: `_write_json`
deletes yesterday's `portfolios_*.json` the instant today's lands, so a slate
nobody captured before the next run of this notebook is simply gone. The inbox
accumulates -- run `06` five times before `07` and five slates are waiting.

In [9]:
if WRITE_WORKBOOK:
    out = fpp.report.write_portfolio_workbook(res, stake=STAKE, source=ODDS_FILE.name)
    print("->", out)

# Capture first, because it is what names the run. Portfolio ids restart at 1
# every run, so `770` stops identifying anything the moment there are two of
# them; the code goes into the payload below and the page prints `R007-770`.
slate = fpp.ledger.capture(res, filled, source=ODDS_FILE.name)
print("->", slate)

# `filled` goes in as well as `res`: it carries every price a book quoted, not
# just the ones that cleared E >= 1, and the Match Board draws a negative edge
# bar wherever the model looked and the price was not there.
port_json = fpp.report.write_portfolios_json(res, filled, source=ODDS_FILE.name,
                                             run_code=slate.run_code)
book = fpp.report.write_edge_book()
print("\n->", book)

-> R012 -> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Outputs/Analysis/Pending/slate_R012_2026-09-11 (835 props, 2360 portfolios)
Wrote 2,360 undominated portfolios of 100,000 scored, 104 propositions (835 priced) -> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Outputs/Data/portfolios_2026-09-11.json
Wrote 7.5 MB -> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Outputs/edge_book_2026-09-11.html

-> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Outputs/edge_book_2026-09-11.html
